In [1]:
# ============================================================
# PS S6E5 - exp_20260501_001b_original_schema_check
# Original dataset schema / distribution / Normalized_TyreLife check
# ============================================================

In [2]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260501_001b_original_schema_check"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    # Add data:
    # aadigupta1601/f1-strategy-dataset-pit-stop-prediction
    ORIG_BASE_CANDIDATES = [
        Path("/kaggle/input/f1-strategy-dataset-pit-stop-prediction"),
        Path("/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction"), # /f1_strategy_dataset_v4.csv
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Load competition data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

print("competition train shape:", train.shape)
print("competition test shape :", test.shape)
print("sample submission shape:", sub.shape)

print("\ncompetition train columns:")
print(train.columns.tolist())

print("\ncompetition test columns:")
print(test.columns.tolist())

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET not in test.columns

comp_feature_cols = [c for c in train.columns if c not in [CFG.ID_COL, CFG.TARGET]]

print("\ncompetition feature columns:", len(comp_feature_cols))
print(comp_feature_cols)

print("\ncompetition target distribution:")
print(train[CFG.TARGET].value_counts(dropna=False))
print(train[CFG.TARGET].value_counts(normalize=True, dropna=False))

competition train shape: (439140, 16)
competition test shape : (188165, 15)
sample submission shape: (188165, 2)

competition train columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']

competition test columns:
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

competition feature columns: 14
['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

competition target distribution:
PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


In [5]:
# ============================================================
# Locate original dataset
# ============================================================

print("\n/kaggle/input tree, csv files:")
all_csvs = sorted(Path("/kaggle/input").glob("**/*.csv"))
for p in all_csvs:
    print(p)

orig_base = None
for p in CFG.ORIG_BASE_CANDIDATES:
    if p.exists():
        orig_base = p
        break

if orig_base is None:
    print("\nCould not find original dataset from predefined paths.")
    print("Trying to infer from csv files...")
    candidate_csvs = []
    for p in all_csvs:
        if "playground-series-s6e5" in str(p):
            continue
        candidate_csvs.append(p)

    print("\nnon-competition csv candidates:")
    for p in candidate_csvs:
        print(p)

    assert len(candidate_csvs) > 0, "Original dataset csv was not found. Add Kaggle dataset first."

    orig_csvs = candidate_csvs
else:
    print("\nDetected original base:", orig_base)
    orig_csvs = sorted(orig_base.glob("*.csv"))

print("\noriginal csv candidates:")
for p in orig_csvs:
    print(p)


/kaggle/input tree, csv files:
/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv

Detected original base: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction

original csv candidates:
/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv


In [6]:
# ============================================================
# Read all original csv candidates lightly
# ============================================================

orig_infos = []

for p in orig_csvs:
    try:
        tmp = pd.read_csv(p, nrows=5)
        cols = tmp.columns.tolist()
        overlap = len((set(train.columns) - {CFG.ID_COL}) & set(cols))

        info = {
            "path": str(p),
            "filename": p.name,
            "ncols": len(cols),
            "columns": cols,
            "overlap_with_train_no_id": overlap,
            "has_target": CFG.TARGET in cols,
            "has_normalized_tyrelife": CFG.DANGER_COL in cols,
        }
        orig_infos.append(info)

        print("\n---", p.name, "---")
        print("ncols:", len(cols))
        print("overlap_with_train_no_id:", overlap)
        print("has_target:", CFG.TARGET in cols)
        print("has_Normalized_TyreLife:", CFG.DANGER_COL in cols)
        print(cols)

    except Exception as e:
        print("failed:", p, e)

orig_info_df = pd.DataFrame(orig_infos)
display(orig_info_df[[
    "filename",
    "ncols",
    "overlap_with_train_no_id",
    "has_target",
    "has_normalized_tyrelife",
    "path",
]])

orig_info_df.to_csv(CFG.OUTDIR / "original_csv_candidates.csv", index=False)


--- f1_strategy_dataset_v4.csv ---
ncols: 16
overlap_with_train_no_id: 15
has_target: True
has_Normalized_TyreLife: True
['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'Race', 'Year', 'LapTime_Delta', 'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress', 'Normalized_TyreLife', 'Position_Change']


,filename,ncols,overlap_with_train_no_id,has_target,has_normalized_tyrelife,path
0,f1_strategy_dataset_v4.csv,16,15,True,True,/kaggle/input/datasets/aadigupta1601/f1-strate...


In [7]:
# ============================================================
# Select original main csv
# ============================================================

# Prefer csv that has target and highest overlap with train columns.
valid = [x for x in orig_infos if x["has_target"]]
if not valid:
    valid = orig_infos

valid = sorted(
    valid,
    key=lambda x: (x["overlap_with_train_no_id"], x["ncols"]),
    reverse=True,
)

assert len(valid) > 0, "No original csv candidate."

orig_path = Path(valid[0]["path"])
print("\nSelected original csv:", orig_path)

orig = pd.read_csv(orig_path)

print("\noriginal shape:", orig.shape)
print("\noriginal columns:")
print(orig.columns.tolist())

print("\noriginal dtypes:")
print(orig.dtypes)


Selected original csv: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv

original shape: (101371, 16)

original columns:
['Driver', 'LapNumber', 'Compound', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'Race', 'Year', 'LapTime_Delta', 'Cumulative_Degradation', 'PitStop', 'PitNextLap', 'RaceProgress', 'Normalized_TyreLife', 'Position_Change']

original dtypes:
Driver                     object
LapNumber                   int64
Compound                   object
Stint                       int64
TyreLife                  float64
Position                    int64
LapTime (s)               float64
Race                       object
Year                        int64
LapTime_Delta             float64
Cumulative_Degradation    float64
PitStop                     int64
PitNextLap                  int64
RaceProgress              float64
Normalized_TyreLife       float64
Position_Change           float64
dtype: object


In [8]:
# ============================================================
# Schema comparison
# ============================================================

train_cols_no_id = set(train.columns) - {CFG.ID_COL}
test_cols_no_id = set(test.columns) - {CFG.ID_COL}
orig_cols = set(orig.columns)

schema_summary = {
    "selected_original_path": str(orig_path),
    "train_shape": list(train.shape),
    "test_shape": list(test.shape),
    "original_shape": list(orig.shape),
    "train_columns": train.columns.tolist(),
    "test_columns": test.columns.tolist(),
    "original_columns": orig.columns.tolist(),
    "train_only_vs_original": sorted(train_cols_no_id - orig_cols),
    "original_only_vs_train": sorted(orig_cols - train_cols_no_id),
    "common_train_original": sorted(train_cols_no_id & orig_cols),
    "test_only_vs_original_features": sorted(test_cols_no_id - (orig_cols - {CFG.TARGET})),
    "original_only_vs_test_features": sorted((orig_cols - {CFG.TARGET}) - test_cols_no_id),
    "has_normalized_tyrelife_in_original": CFG.DANGER_COL in orig.columns,
    "has_normalized_tyrelife_in_train": CFG.DANGER_COL in train.columns,
    "has_normalized_tyrelife_in_test": CFG.DANGER_COL in test.columns,
}

print("\ntrain only vs original:")
print(schema_summary["train_only_vs_original"])

print("\noriginal only vs train:")
print(schema_summary["original_only_vs_train"])

print("\ncommon train/original:")
print(schema_summary["common_train_original"])

print("\ntest only vs original features:")
print(schema_summary["test_only_vs_original_features"])

print("\noriginal only vs test features:")
print(schema_summary["original_only_vs_test_features"])

with open(CFG.OUTDIR / "schema_summary.json", "w", encoding="utf-8") as f:
    json.dump(schema_summary, f, ensure_ascii=False, indent=2)

pd.DataFrame({
    "train_only_vs_original": pd.Series(schema_summary["train_only_vs_original"]),
}).to_csv(CFG.OUTDIR / "train_only_vs_original.csv", index=False)

pd.DataFrame({
    "original_only_vs_train": pd.Series(schema_summary["original_only_vs_train"]),
}).to_csv(CFG.OUTDIR / "original_only_vs_train.csv", index=False)


train only vs original:
[]

original only vs train:
['Normalized_TyreLife']

common train/original:
['Compound', 'Cumulative_Degradation', 'Driver', 'LapNumber', 'LapTime (s)', 'LapTime_Delta', 'PitNextLap', 'PitStop', 'Position', 'Position_Change', 'Race', 'RaceProgress', 'Stint', 'TyreLife', 'Year']

test only vs original features:
[]

original only vs test features:
['Normalized_TyreLife']


In [9]:
# ============================================================
# Dtype / missing / nunique comparison
# ============================================================

common_cols = sorted(train_cols_no_id & orig_cols)

rows = []
for c in common_cols:
    rows.append({
        "column": c,
        "train_dtype": str(train[c].dtype),
        "test_dtype": str(test[c].dtype) if c in test.columns else None,
        "orig_dtype": str(orig[c].dtype),
        "train_missing": int(train[c].isna().sum()),
        "test_missing": int(test[c].isna().sum()) if c in test.columns else None,
        "orig_missing": int(orig[c].isna().sum()),
        "train_nunique": int(train[c].nunique(dropna=False)),
        "test_nunique": int(test[c].nunique(dropna=False)) if c in test.columns else None,
        "orig_nunique": int(orig[c].nunique(dropna=False)),
    })

dtype_missing_df = pd.DataFrame(rows)
display(dtype_missing_df)
dtype_missing_df.to_csv(CFG.OUTDIR / "dtype_missing_nunique_comparison.csv", index=False)

,column,train_dtype,test_dtype,orig_dtype,train_missing,test_missing,orig_missing,train_nunique,test_nunique,orig_nunique
0,Compound,object,object,object,0,0.0,66,5,5.0,6
1,Cumulative_Degradation,float64,float64,float64,0,0.0,0,142701,86823.0,81360
2,Driver,object,object,object,0,0.0,0,887,801.0,31
3,LapNumber,int64,int64,int64,0,0.0,0,78,77.0,78
4,LapTime (s),float64,float64,float64,0,0.0,0,37719,30286.0,40779
5,LapTime_Delta,float64,float64,float64,0,0.0,0,57532,40673.0,54084
6,PitNextLap,float64,None,int64,0,NaN,0,2,NaN,2
7,PitStop,int64,int64,int64,0,0.0,0,2,2.0,2
8,Position,int64,int64,int64,0,0.0,0,20,20.0,20
9,Position_Change,float64,float64,float64,0,0.0,0,37,37.0,37


In [10]:
# ============================================================
# Value counts for categorical-like columns
# ============================================================

def is_categorical_like(s: pd.Series, max_unique: int = 80) -> bool:
    if s.dtype == "object" or str(s.dtype) == "category" or str(s.dtype) == "bool":
        return True
    if s.nunique(dropna=False) <= max_unique:
        return True
    return False


cat_like_cols = [c for c in common_cols if is_categorical_like(train[c]) or is_categorical_like(orig[c])]
print("\ncategorical-like columns:", cat_like_cols)

value_count_rows = []

for c in cat_like_cols:
    train_vc = train[c].value_counts(dropna=False, normalize=True)
    test_vc = test[c].value_counts(dropna=False, normalize=True) if c in test.columns else pd.Series(dtype=float)
    orig_vc = orig[c].value_counts(dropna=False, normalize=True)

    all_vals = sorted(
        set(train_vc.index.astype(str)) |
        set(test_vc.index.astype(str)) |
        set(orig_vc.index.astype(str))
    )

    # Reindex through stringified value keys for stable output.
    train_map = {str(k): v for k, v in train_vc.items()}
    test_map = {str(k): v for k, v in test_vc.items()}
    orig_map = {str(k): v for k, v in orig_vc.items()}

    for v in all_vals:
        value_count_rows.append({
            "column": c,
            "value": v,
            "train_rate": float(train_map.get(v, 0.0)),
            "test_rate": float(test_map.get(v, 0.0)),
            "orig_rate": float(orig_map.get(v, 0.0)),
            "abs_train_orig_diff": abs(float(train_map.get(v, 0.0)) - float(orig_map.get(v, 0.0))),
            "abs_test_orig_diff": abs(float(test_map.get(v, 0.0)) - float(orig_map.get(v, 0.0))),
        })

value_count_df = pd.DataFrame(value_count_rows)
display(value_count_df.head(100))
value_count_df.to_csv(CFG.OUTDIR / "categorical_value_rate_comparison.csv", index=False)

# Top drift values
if len(value_count_df) > 0:
    display(
        value_count_df
        .sort_values("abs_train_orig_diff", ascending=False)
        .head(50)
    )


categorical-like columns: ['Compound', 'Driver', 'LapNumber', 'PitNextLap', 'PitStop', 'Position', 'Position_Change', 'Race', 'Stint', 'TyreLife', 'Year']


,column,value,train_rate,test_rate,orig_rate,abs_train_orig_diff,abs_test_orig_diff
0,Compound,HARD,0.388300,0.386241,0.443588,0.055289,0.057348
1,Compound,INTERMEDIATE,0.039582,0.039370,0.054809,0.015227,0.015439
2,Compound,MEDIUM,0.480806,0.483071,0.371299,0.109506,0.111771
3,Compound,SOFT,0.088227,0.088300,0.125716,0.037489,0.037416
4,Compound,WET,0.003086,0.003019,0.003936,0.000850,0.000917
...,...,...,...,...,...,...,...
95,Driver,D064,0.002887,0.003029,0.000000,0.002887,0.003029
96,Driver,D065,0.002862,0.002833,0.000000,0.002862,0.002833
97,Driver,D066,0.002794,0.002923,0.000000,0.002794,0.002923
98,Driver,D067,0.002714,0.002907,0.000000,0.002714,0.002907


,column,value,train_rate,test_rate,orig_rate,abs_train_orig_diff,abs_test_orig_diff
972,PitNextLap,0.0,0.801018,0.000000,0.000000,0.801018,0.000000
971,PitNextLap,0,0.000000,0.000000,0.745203,0.745203,0.745203
973,PitNextLap,1,0.000000,0.000000,0.254797,0.254797,0.254797
974,PitNextLap,1.0,0.198982,0.000000,0.000000,0.198982,0.000000
1062,Stint,1,0.492526,0.495246,0.315100,0.177426,0.180146
1063,Stint,2,0.294977,0.294050,0.411084,0.116107,0.117034
976,PitStop,1,0.136118,0.136263,0.251581,0.115463,0.115317
975,PitStop,0,0.863882,0.863737,0.748419,0.115463,0.115317
2,Compound,MEDIUM,0.480806,0.483071,0.371299,0.109506,0.111771
1150,Year,2023,0.310031,0.309090,0.245770,0.064260,0.063320


In [11]:
# ============================================================
# Numeric distribution comparison
# ============================================================

numeric_cols = [
    c for c in common_cols
    if pd.api.types.is_numeric_dtype(train[c]) and pd.api.types.is_numeric_dtype(orig[c])
]

print("\nnumeric common columns:", numeric_cols)

num_rows = []
for c in numeric_cols:
    tr = train[c]
    te = test[c] if c in test.columns else None
    og = orig[c]

    row = {
        "column": c,
        "train_mean": float(tr.mean()),
        "train_std": float(tr.std()),
        "train_min": float(tr.min()),
        "train_q01": float(tr.quantile(0.01)),
        "train_q05": float(tr.quantile(0.05)),
        "train_q25": float(tr.quantile(0.25)),
        "train_median": float(tr.median()),
        "train_q75": float(tr.quantile(0.75)),
        "train_q95": float(tr.quantile(0.95)),
        "train_q99": float(tr.quantile(0.99)),
        "train_max": float(tr.max()),
        "orig_mean": float(og.mean()),
        "orig_std": float(og.std()),
        "orig_min": float(og.min()),
        "orig_q01": float(og.quantile(0.01)),
        "orig_q05": float(og.quantile(0.05)),
        "orig_q25": float(og.quantile(0.25)),
        "orig_median": float(og.median()),
        "orig_q75": float(og.quantile(0.75)),
        "orig_q95": float(og.quantile(0.95)),
        "orig_q99": float(og.quantile(0.99)),
        "orig_max": float(og.max()),
    }

    if te is not None and pd.api.types.is_numeric_dtype(te):
        row.update({
            "test_mean": float(te.mean()),
            "test_std": float(te.std()),
            "test_min": float(te.min()),
            "test_q01": float(te.quantile(0.01)),
            "test_q05": float(te.quantile(0.05)),
            "test_q25": float(te.quantile(0.25)),
            "test_median": float(te.median()),
            "test_q75": float(te.quantile(0.75)),
            "test_q95": float(te.quantile(0.95)),
            "test_q99": float(te.quantile(0.99)),
            "test_max": float(te.max()),
        })

    row["mean_abs_train_orig_diff"] = abs(row["train_mean"] - row["orig_mean"])
    row["median_abs_train_orig_diff"] = abs(row["train_median"] - row["orig_median"])
    row["std_abs_train_orig_diff"] = abs(row["train_std"] - row["orig_std"])

    num_rows.append(row)

num_dist_df = pd.DataFrame(num_rows)
display(num_dist_df)
num_dist_df.to_csv(CFG.OUTDIR / "numeric_distribution_comparison.csv", index=False)

if len(num_dist_df) > 0:
    display(
        num_dist_df
        .sort_values("mean_abs_train_orig_diff", ascending=False)
        [["column", "train_mean", "orig_mean", "mean_abs_train_orig_diff", "train_std", "orig_std", "std_abs_train_orig_diff"]]
    )


numeric common columns: ['Cumulative_Degradation', 'LapNumber', 'LapTime (s)', 'LapTime_Delta', 'PitNextLap', 'PitStop', 'Position', 'Position_Change', 'RaceProgress', 'Stint', 'TyreLife', 'Year']


,column,train_mean,train_std,train_min,train_q01,train_q05,train_q25,train_median,train_q75,train_q95,...,test_q05,test_q25,test_median,test_q75,test_q95,test_q99,test_max,mean_abs_train_orig_diff,median_abs_train_orig_diff,std_abs_train_orig_diff
0,Cumulative_Degradation,-25.721759,54.766573,-274.564000,-205.034000,-104.789250,-46.56625,-20.994000,-6.199000,84.401000,...,-105.052400,-46.814000,-21.027000,-6.188000,84.008400,121.999000,2406.800,3.828292,0.684000,15.469186
1,LapNumber,23.105909,16.958261,1.000000,1.000000,2.000000,9.00000,19.000000,36.000000,54.000000,...,2.000000,9.000000,19.000000,36.000000,54.000000,66.000000,77.000,7.338932,11.000000,1.188681
2,LapTime (s),90.948735,19.772769,67.694000,70.723000,75.053000,82.62100,90.521000,98.471000,109.726050,...,75.091000,82.634000,90.485000,98.485000,109.800000,124.940000,2497.905,1.638453,0.646000,13.458645
3,LapTime_Delta,-3.770040,43.945759,-2403.895000,-40.256880,-24.819000,-8.88400,-0.295000,0.115000,19.026000,...,-24.654000,-8.874000,-0.285000,0.122000,18.974000,30.534000,2433.472,3.566150,0.268000,1.399151
4,PitNextLap,0.198982,0.399235,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.055815,0.000000,0.036514
5,PitStop,0.136118,0.342915,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000,0.115463,0.000000,0.091009
6,Position,9.630339,5.278770,1.000000,1.000000,1.000000,5.00000,10.000000,14.000000,18.000000,...,1.000000,5.000000,10.000000,14.000000,18.000000,20.000000,20.000,0.128793,0.000000,0.127686
7,Position_Change,0.101542,4.006765,-18.000000,-11.000000,-7.000000,-1.00000,0.000000,2.000000,7.000000,...,-7.000000,-1.000000,0.000000,2.000000,7.000000,11.000000,18.000,0.106178,0.000000,0.094039
8,RaceProgress,0.337661,0.253277,0.012821,0.013158,0.027778,0.12987,0.269231,0.513158,0.842105,...,0.027778,0.128205,0.269231,0.512821,0.840909,0.948276,1.000,0.094958,0.151822,0.004851
9,Stint,1.789113,0.950194,1.000000,1.000000,1.000000,1.00000,2.000000,2.000000,4.000000,...,1.000000,1.000000,2.000000,2.000000,4.000000,5.000000,8.000,0.257281,0.000000,0.001397


,column,train_mean,orig_mean,mean_abs_train_orig_diff,train_std,orig_std,std_abs_train_orig_diff
1,LapNumber,23.105909,30.444841,7.338932,16.958261,18.146942,1.188681
0,Cumulative_Degradation,-25.721759,-29.550051,3.828292,54.766573,70.235759,15.469186
3,LapTime_Delta,-3.770040,-0.203891,3.566150,43.945759,45.344910,1.399151
2,LapTime (s),90.948735,92.587188,1.638453,19.772769,33.231414,13.458645
10,TyreLife,14.158231,14.549339,0.391108,9.801338,10.313385,0.512047
9,Stint,1.789113,2.046394,0.257281,0.950194,0.948797,0.001397
6,Position,9.630339,9.759132,0.128793,5.278770,5.406456,0.127686
5,PitStop,0.136118,0.251581,0.115463,0.342915,0.433924,0.091009
7,Position_Change,0.101542,-0.004636,0.106178,4.006765,3.912725,0.094039
8,RaceProgress,0.337661,0.432618,0.094958,0.253277,0.258129,0.004851


In [12]:
# ============================================================
# Target distribution comparison
# ============================================================

target_summary = {}

if CFG.TARGET in orig.columns:
    print("\ncompetition target distribution:")
    print(train[CFG.TARGET].value_counts(dropna=False))
    print(train[CFG.TARGET].value_counts(normalize=True, dropna=False))

    print("\noriginal target distribution:")
    print(orig[CFG.TARGET].value_counts(dropna=False))
    print(orig[CFG.TARGET].value_counts(normalize=True, dropna=False))

    target_summary = {
        "competition_train_counts": {
            str(k): int(v)
            for k, v in train[CFG.TARGET].value_counts(dropna=False).to_dict().items()
        },
        "competition_train_rates": {
            str(k): float(v)
            for k, v in train[CFG.TARGET].value_counts(normalize=True, dropna=False).to_dict().items()
        },
        "original_counts": {
            str(k): int(v)
            for k, v in orig[CFG.TARGET].value_counts(dropna=False).to_dict().items()
        },
        "original_rates": {
            str(k): float(v)
            for k, v in orig[CFG.TARGET].value_counts(normalize=True, dropna=False).to_dict().items()
        },
    }

with open(CFG.OUTDIR / "target_distribution_summary.json", "w", encoding="utf-8") as f:
    json.dump(target_summary, f, ensure_ascii=False, indent=2)


competition target distribution:
PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64

original target distribution:
PitNextLap
0    75542
1    25829
Name: count, dtype: int64
PitNextLap
0    0.745203
1    0.254797
Name: proportion, dtype: float64


In [13]:
# ============================================================
# Original priors by common categorical-like columns
# ============================================================

prior_rows = []

if CFG.TARGET in orig.columns:
    for c in cat_like_cols:
        if c == CFG.TARGET:
            continue

        g = orig.groupby(c, dropna=False)[CFG.TARGET].agg(["count", "mean"]).reset_index()
        g = g.rename(columns={"count": "orig_count", "mean": "orig_target_mean"})

        train_g = train.groupby(c, dropna=False)[CFG.TARGET].agg(["count", "mean"]).reset_index()
        train_g = train_g.rename(columns={"count": "train_count", "mean": "train_target_mean"})

        merged = g.merge(train_g, on=c, how="outer")
        merged["column"] = c
        merged = merged.rename(columns={c: "value"})
        merged["value"] = merged["value"].astype(str)
        merged["abs_mean_diff"] = (merged["orig_target_mean"] - merged["train_target_mean"]).abs()

        prior_rows.append(merged[[
            "column",
            "value",
            "orig_count",
            "orig_target_mean",
            "train_count",
            "train_target_mean",
            "abs_mean_diff",
        ]])

if prior_rows:
    prior_df = pd.concat(prior_rows, axis=0, ignore_index=True)
    display(prior_df.sort_values(["column", "orig_count"], ascending=[True, False]).head(100))
    prior_df.to_csv(CFG.OUTDIR / "original_vs_train_target_priors_by_category.csv", index=False)

    print("\nTop prior differences:")
    display(prior_df.sort_values("abs_mean_diff", ascending=False).head(50))
else:
    prior_df = pd.DataFrame()

,column,value,orig_count,orig_target_mean,train_count,train_target_mean,abs_mean_diff
0,Compound,HARD,44967.0,0.321903,170518.0,0.327537,0.005635
2,Compound,MEDIUM,37639.0,0.190760,211141.0,0.101131,0.089628
3,Compound,SOFT,12744.0,0.239093,38744.0,0.193475,0.045618
1,Compound,INTERMEDIATE,5556.0,0.194744,17382.0,0.152284,0.042460
4,Compound,WET,399.0,0.017544,1355.0,0.025092,0.007548
...,...,...,...,...,...,...,...
71,Driver,D040,NaN,NaN,1327.0,0.200452,NaN
72,Driver,D041,NaN,NaN,1363.0,0.185620,NaN
73,Driver,D042,NaN,NaN,1352.0,0.200444,NaN
74,Driver,D043,NaN,NaN,1312.0,0.190549,NaN



Top prior differences:


,column,value,orig_count,orig_target_mean,train_count,train_target_mean,abs_mean_diff
970,LapNumber,78,20.0,0.550000,2.0,0.000000,0.550000
1140,TyreLife,74.0,10.0,0.000000,27.0,0.444444,0.444444
993,Position_Change,-18.0,21.0,0.666667,113.0,0.230088,0.436578
1029,Position_Change,18.0,1.0,0.000000,3.0,0.333333,0.333333
1143,TyreLife,77.0,6.0,0.666667,3.0,0.333333,0.333333
994,Position_Change,-17.0,12.0,0.583333,63.0,0.269841,0.313492
1027,Position_Change,16.0,19.0,0.631579,94.0,0.393617,0.237962
1138,TyreLife,72.0,11.0,0.272727,19.0,0.473684,0.200957
882,Driver,VET,1022.0,0.374755,359.0,0.565460,0.190704
1135,TyreLife,69.0,13.0,0.923077,30.0,0.733333,0.189744


In [14]:
# ============================================================
# Normalized_TyreLife danger analysis
# ============================================================

danger_summary = {
    "has_normalized_tyrelife": CFG.DANGER_COL in orig.columns,
    "used_in_model": False,
    "recommendation": "Do not use in mainline. Treat as dangerous/oracle-like column.",
}

if CFG.DANGER_COL in orig.columns:
    print("\nNormalized_TyreLife exists in original dataset.")
    print(orig[CFG.DANGER_COL].describe())
    print("\nMissing:", orig[CFG.DANGER_COL].isna().sum())
    print("Nunique:", orig[CFG.DANGER_COL].nunique(dropna=False))

    # Correlation with target
    if CFG.TARGET in orig.columns and pd.api.types.is_numeric_dtype(orig[CFG.DANGER_COL]):
        tmp = orig[[CFG.DANGER_COL, CFG.TARGET]].dropna()
        if len(tmp) > 0:
            corr = tmp[CFG.DANGER_COL].corr(tmp[CFG.TARGET])
            danger_summary["corr_with_target"] = None if pd.isna(corr) else float(corr)

            # Target rate by bins
            try:
                orig["_NTL_bin"] = pd.qcut(orig[CFG.DANGER_COL], q=10, duplicates="drop")
                ntl_bin = (
                    orig
                    .groupby("_NTL_bin", observed=True)[CFG.TARGET]
                    .agg(["count", "mean"])
                    .reset_index()
                )
                ntl_bin["_NTL_bin"] = ntl_bin["_NTL_bin"].astype(str)
                display(ntl_bin)
                ntl_bin.to_csv(CFG.OUTDIR / "normalized_tyrelife_target_rate_by_bin.csv", index=False)
                danger_summary["bin_table_created"] = True
            except Exception as e:
                print("qcut failed:", e)
                danger_summary["bin_table_created"] = False

    # Simple relationship with TyreLife
    if "TyreLife" in orig.columns:
        tmp = orig[[CFG.DANGER_COL, "TyreLife"]].dropna()
        if len(tmp) > 0:
            danger_summary["corr_with_TyreLife"] = float(tmp[CFG.DANGER_COL].corr(tmp["TyreLife"]))

    if "Stint" in orig.columns and "TyreLife" in orig.columns:
        # This checks whether Normalized_TyreLife is likely tied to within-stint scaling.
        # It is EDA only. Do not port this as feature generation.
        grp_cols = ["Race", "Driver", "Stint"]
        if all(c in orig.columns for c in grp_cols):
            tmp = orig.copy()
            tmp["_stint_max_TyreLife"] = tmp.groupby(grp_cols)["TyreLife"].transform("max")
            tmp["_naive_TyreLife_div_stint_max"] = tmp["TyreLife"] / tmp["_stint_max_TyreLife"].replace(0, np.nan)
            corr_proxy = tmp[[CFG.DANGER_COL, "_naive_TyreLife_div_stint_max"]].dropna().corr().iloc[0, 1]
            danger_summary["corr_with_naive_TyreLife_div_stint_max_by_Race_Driver_Stint"] = float(corr_proxy)

            print("\nCorrelation with naive TyreLife / stint max by Race-Driver-Stint:")
            print(corr_proxy)

            # Save small sample for inspection
            tmp_sample = tmp[
                grp_cols + ["LapNumber", "TyreLife", "_stint_max_TyreLife", "_naive_TyreLife_div_stint_max", CFG.DANGER_COL, CFG.TARGET]
            ].head(200)
            tmp_sample.to_csv(CFG.OUTDIR / "normalized_tyrelife_proxy_inspection_sample.csv", index=False)

else:
    print("\nNormalized_TyreLife does not exist in selected original csv.")

with open(CFG.OUTDIR / "normalized_tyrelife_danger_summary.json", "w", encoding="utf-8") as f:
    json.dump(danger_summary, f, ensure_ascii=False, indent=2)

print("\nDanger summary:")
print(json.dumps(danger_summary, ensure_ascii=False, indent=2))


Normalized_TyreLife exists in original dataset.
count    101371.000000
mean          0.386521
std           0.259906
min           0.012821
25%           0.172414
50%           0.333333
75%           0.562500
max           1.000000
Name: Normalized_TyreLife, dtype: float64

Missing: 0
Nunique: 1340


,_NTL_bin,count,mean
0,"(0.011800000000000001, 0.08]",10198,0.064130
1,"(0.08, 0.143]",10511,0.164114
2,"(0.143, 0.204]",9720,0.219753
3,"(0.204, 0.267]",10133,0.265272
4,"(0.267, 0.333]",10340,0.295648
5,"(0.333, 0.415]",9930,0.318630
6,"(0.415, 0.5]",10221,0.319832
7,"(0.5, 0.625]",10128,0.328397
8,"(0.625, 0.786]",10062,0.293480
9,"(0.786, 1.0]",10128,0.282089



Correlation with naive TyreLife / stint max by Race-Driver-Stint:
0.769469377186573

Danger summary:
{
  "has_normalized_tyrelife": true,
  "used_in_model": false,
  "recommendation": "Do not use in mainline. Treat as dangerous/oracle-like column.",
  "corr_with_target": 0.12539850080038023,
  "bin_table_created": true,
  "corr_with_TyreLife": 0.7752929258695356,
  "corr_with_naive_TyreLife_div_stint_max_by_Race_Driver_Stint": 0.769469377186573
}


In [15]:
# ============================================================
# Train/test/original row overlap rough check
# ============================================================

# Exact row overlap is not expected because train/test are synthetic-inspired.
# Still useful to confirm whether original rows are literally copied.
overlap_features = [c for c in comp_feature_cols if c in orig.columns]

print("\noverlap_features for exact duplicate check:", overlap_features)

def make_row_hash(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    # Use string representation for rough duplicate check.
    return pd.util.hash_pandas_object(
        df[cols].astype(str),
        index=False
    )

train_hash = make_row_hash(train, overlap_features)
test_hash = make_row_hash(test, overlap_features)
orig_hash = make_row_hash(orig, overlap_features)

train_orig_overlap = int(train_hash.isin(set(orig_hash)).sum())
test_orig_overlap = int(test_hash.isin(set(orig_hash)).sum())
train_test_overlap = int(train_hash.isin(set(test_hash)).sum())

overlap_summary = {
    "overlap_features": overlap_features,
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "original_rows": int(len(orig)),
    "exact_train_rows_found_in_original": train_orig_overlap,
    "exact_test_rows_found_in_original": test_orig_overlap,
    "exact_train_rows_found_in_test": train_test_overlap,
    "exact_train_rows_found_in_original_rate": float(train_orig_overlap / len(train)),
    "exact_test_rows_found_in_original_rate": float(test_orig_overlap / len(test)),
    "exact_train_rows_found_in_test_rate": float(train_test_overlap / len(train)),
}

print("\nOverlap summary:")
print(json.dumps(overlap_summary, ensure_ascii=False, indent=2))

with open(CFG.OUTDIR / "row_overlap_summary.json", "w", encoding="utf-8") as f:
    json.dump(overlap_summary, f, ensure_ascii=False, indent=2)


overlap_features for exact duplicate check: ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

Overlap summary:
{
  "overlap_features": [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change"
  ],
  "train_rows": 439140,
  "test_rows": 188165,
  "original_rows": 101371,
  "exact_train_rows_found_in_original": 0,
  "exact_test_rows_found_in_original": 0,
  "exact_train_rows_found_in_test": 0,
  "exact_train_rows_found_in_original_rate": 0.0,
  "exact_test_rows_found_in_original_rate": 0.0,
  "exact_train_rows_found_in_test_rate": 0.0
}


In [16]:
# ============================================================
# Final report
# ============================================================

report = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "purpose": "original schema check",
    },
    "selected_original_path": str(orig_path),
    "schema_summary": schema_summary,
    "target_summary": target_summary,
    "danger_summary": danger_summary,
    "overlap_summary": overlap_summary,
    "output_files": {
        "schema_summary": str(CFG.OUTDIR / "schema_summary.json"),
        "original_csv_candidates": str(CFG.OUTDIR / "original_csv_candidates.csv"),
        "dtype_missing_nunique_comparison": str(CFG.OUTDIR / "dtype_missing_nunique_comparison.csv"),
        "categorical_value_rate_comparison": str(CFG.OUTDIR / "categorical_value_rate_comparison.csv"),
        "numeric_distribution_comparison": str(CFG.OUTDIR / "numeric_distribution_comparison.csv"),
        "target_distribution_summary": str(CFG.OUTDIR / "target_distribution_summary.json"),
        "original_vs_train_target_priors_by_category": str(CFG.OUTDIR / "original_vs_train_target_priors_by_category.csv"),
        "normalized_tyrelife_danger_summary": str(CFG.OUTDIR / "normalized_tyrelife_danger_summary.json"),
        "row_overlap_summary": str(CFG.OUTDIR / "row_overlap_summary.json"),
    },
    "decision_notes": [
        "This notebook does not train a model.",
        "Use this result to decide whether original prior min is safe.",
        "Do not use Normalized_TyreLife in mainline.",
        "If original schema is train + Normalized_TyreLife only, original prior min without Normalized_TyreLife is reasonable.",
        "If train/original distribution drift is large, original priors should be tested carefully and isolated.",
    ],
}

with open(CFG.OUTDIR / "check_result.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\nSaved check artifacts to:", CFG.OUTDIR)

print("\nFiles:")
for p in sorted(CFG.OUTDIR.glob("*")):
    print(p)


Saved check artifacts to: /kaggle/working/exp_20260501_001b_original_schema_check

Files:
/kaggle/working/exp_20260501_001b_original_schema_check/categorical_value_rate_comparison.csv
/kaggle/working/exp_20260501_001b_original_schema_check/check_result.json
/kaggle/working/exp_20260501_001b_original_schema_check/dtype_missing_nunique_comparison.csv
/kaggle/working/exp_20260501_001b_original_schema_check/normalized_tyrelife_danger_summary.json
/kaggle/working/exp_20260501_001b_original_schema_check/normalized_tyrelife_proxy_inspection_sample.csv
/kaggle/working/exp_20260501_001b_original_schema_check/normalized_tyrelife_target_rate_by_bin.csv
/kaggle/working/exp_20260501_001b_original_schema_check/numeric_distribution_comparison.csv
/kaggle/working/exp_20260501_001b_original_schema_check/original_csv_candidates.csv
/kaggle/working/exp_20260501_001b_original_schema_check/original_only_vs_train.csv
/kaggle/working/exp_20260501_001b_original_schema_check/original_vs_train_target_priors_by